In [1]:
# EXPERIMENT 1 : COMPARING MULTITHREADING WITH SINGLE THREAD EXECUTION WHEN I/O TASK IS INVOLVED

# single thread example
# these all code blocks are execting line by line in single thread

import time
from threading import Thread


def ask_user():
    start = time.time() # stores the current time
    input("enter name : ")
    print(f"ask user time : {time.time()-start}")
    print()


def complex_calculation():
    start = time.time()
    print("starting calculation.... ")
    [x*2 for x in range(100000000)]
    print(f"complex cal... time : {time.time() - start}")
    print()

start = time.time()
ask_user()
complex_calculation()
print(f"single thread total time : {time.time()-start}"); print()



thread1 = Thread(target=complex_calculation)
thread2 = Thread(target=ask_user)

start = time.time()
thread1.start()
thread2.start()

thread1.join()
thread2.join()

print(f"two thread total time : {time.time()-start}"); print()


# CLEARLY MULTITHREADING IS WINNING HERE, COZ WHEN THREAD2 WAS WAITING FOR YOUR INPUT, THREAD1 TAKE THE LOCK AND STARTED THE CALCULATION WORK AND HENCE REDUCES THE OVERALL TIME OF EXECUTION


ask user time : 3.8657100200653076

starting calculation.... 
complex cal... time : 6.4091949462890625

single thread total time : 10.276050806045532

starting calculation.... 
ask user time : 2.8101580142974854

complex cal... time : 8.403464794158936

two thread total time : 8.406363010406494



---
---
## 📊 Time Reduction Breakdown: Single-Thread vs. Multi-Thread


```text
ask user time : 2.90 seconds
starting calculation.... 
complex cal... time : 4.13 seconds
single thread total time : 7.03 seconds
```

### 1. The Single-Thread Execution (Total Time: ~7.03 seconds)
In a single-thread model, the **Main Thread** handles tasks strictly one after the other, in a linear sequence:



* **`ask_user()` runs first:** The program completely freezes for **2.90 seconds** doing absolutely nothing but waiting for you to type your name and hit Enter. The CPU sits completely idle during this time.
* **`complex_calculation()` runs second:** Only *after* you hit Enter does the thread move to the math operations, which takes **4.13 seconds** of raw CPU computation.

$$\text{Total Time} = 2.90\text{s (Waiting)} + 4.13\text{s (Calculating)} = 7.03\text{s}$$

---

### 2. The Two-Thread Execution (Total Time: ~4.00 seconds)
When you wrapped those functions inside `thread1` and `thread2` and called `.start()`, you unlocked **Concurrency**. Instead of running back-to-back, both tasks executed simultaneously during the exact same window of time:



* **Overlapping the Idle Time:** The exact millisecond `thread2` opened the input prompt and started waiting for your keyboard entry, `thread1` grabbed the CPU and immediately began processing the heavy 4-second math calculation.
* **The 4-Second Window:** While you were taking 3.99 seconds to think and type your name, the computer was furiously crunching numbers in the background. The heavy math calculation completed at almost the exact same microsecond you hit the Enter key.

Because the idle waiting time and the heavy math calculations happened *at the same time*, the overall time of your program is no longer the sum of both tasks. Instead, it is simply determined by whichever task took the longest to finish.

$$\text{Total Time} = \max(3.99\text{s}, 4.00\text{s}) = 4.00\text{s}$$

---

### 💡 Why the GIL Didn't Ruin This CPU-Bound Task
Earlier, we learned that Python's GIL slows down multi-threaded math calculations. So why did it work perfectly here?

It worked because **one of your tasks was I/O-Bound (`input`) and the other was CPU-Bound (`complex_calculation`)**:

* The moment `thread2` called `input()`, Python **forcibly stripped the GIL away from it** because it was just waiting for a human.
* This allowed `thread1` to grab the GIL instantly and use 100% of the CPU's processing core to handle the heavy math completely uninterrupted.

This proves that multi-threading in Python is incredibly powerful whenever you mix background operations with human interaction, disk storage access, or internet downloads!

---
---

# 🧵 Understanding Thread Synchronization: What does `.join()` do?

To understand `thread1.join()` and `thread2.join()`, we need to look at how your main program coordinates with its background workers.

---

## 1. The Core Concept: The Main Thread vs. Background Threads

When you run a Python script, your operating system creates one primary worker thread called the **Main Thread**. 

In your code, the Main Thread reads through your script from top to bottom. When it hits these lines:
```python
thread1 = Thread(target=complex_calculation)
thread2 = Thread(target=ask_user)

thread1.start()
thread2.start()
```
The Main Thread spawns two independent background worker threads (thread1 and thread2), tells them to start running their respective functions, and immediately moves on to the next line of code. The Main Thread does not wait for them to finish before moving down the script.

## The Mechanics of `.join()`

### 1. Spawning the Background Workers
When you run the multi-threaded code, the **Main Thread** reads down your script. When it encounters `.start()`, it spawns two independent background worker threads (`thread1` and `thread2`), tells them to start running their respective functions, and **immediately moves on to the next line of code**. The Main Thread does *not* wait for them to finish before moving down the script.

---

### 2. What exactly is `.join()` doing?
The word **`join`** means *"force the Main Thread to pause and wait right here until this specific background thread is completely finished."*



Here is the step-by-step trace of how the Main Thread executes your code with the join commands:

* **`thread1.start()` and `thread2.start()`:** The Main Thread kicks off both workers. They are now running concurrently in the background.
* **`thread1.join()`:** The Main Thread hits this line and stops dead in its tracks. It sits there patiently waiting. Even if `thread2` finishes early, or if other things happen, the Main Thread will not move an inch until `thread1` (the complex calculation) finishes 100% of its work.
* **`thread1` finishes:** The Main Thread wakes up and moves to the next line.
* **`thread2.join()`:** The Main Thread pauses again, checking if `thread2` (the user input) is finished. If you are still typing your name, the Main Thread will wait right here.
* **`thread2` finishes:** The Main Thread moves past the guardrail and finally hits the last line:

```python
print(f"two thread total time : {time.time()-start}")

In [2]:
# EXPERIMENT 2 : comparing multithreading with single thread when only cpu bound tasks are involved

# these all code blocks are execting line by line in single thread

import time
from threading import Thread


def complex_calculation():
    start = time.time()
    print("starting calculation.... ")
    [x*2 for x in range(100000000)]
    print(f"complex cal... time : {time.time() - start}")
    print()

start = time.time()
complex_calculation()
complex_calculation()
print(f"single thread total time : {time.time()-start}")



thread1 = Thread(target=complex_calculation)
thread2 = Thread(target=complex_calculation)

start = time.time()
thread1.start()
thread2.start()

thread1.join() 
thread2.join()

print(f"two thread total time : {time.time()-start}")

starting calculation.... 
complex cal... time : 7.95366096496582

starting calculation.... 
complex cal... time : 6.9060587882995605

single thread total time : 14.862847089767456
starting calculation.... 
starting calculation.... 
complex cal... time : 12.750515937805176complex cal... time : 15.008037805557251



two thread total time : 15.010299921035767


---
---

# 🔒 Running Two CPU-Bound Threads: Independence vs. The GIL

In this modified example, you changed both `thread1` and `thread2` to execute `complex_calculation()`. Because both background workers are now fighting for raw computing power (CPU-bound) at the same time, Python's behavior changes dramatically.

---

## 1. The Short Answer: Do they run independently or one after another?

The answer is a mix of both: **They run concurrently (by taking turns), but they CANNOT work truly independently on separate cores.**

To a human eye, they will appear to run at the same time because both processes start and print their execution messages together. However, under the hood, **they are strictly running one after another in tiny, microscopic bursts.**

---

## 2. Why they cannot run truly independently (The GIL Battle)

Because you are using the `threading` module, both threads are trapped inside the exact same Python kitchen building, and they must share the **Global Interpreter Lock (GIL)**.



Here is the microscopic timeline of what happens when you hit execute:
1. **`thread1.start()` & `thread2.start()`:** Both cooks jump into the kitchen.
2. **The Bottleneck:** `thread1` grabs the GIL and begins calculating the massive list on Core 1. `thread2` wants to compute on Core 2, but because it cannot get the GIL, Core 2 sits completely idle.
3. **Time Slicing Intervention:** After a few milliseconds, the Operating System's *Time Slicing* clock alarms. Python forces `thread1` to freeze and drop the GIL.
4. **The Hand-off:** `thread2` immediately snatches the GIL and gets to do its math for a few milliseconds while `thread1` sits and waits.
5. **The Loop:** They continuously pass the single lock back and forth millions of times until the work is done.

---

## 3. The Performance Impact (What your stopwatch will show)

If you look at the total time printed at the bottom of this execution, you will see a frustrating result: **The multi-threaded version will take the same amount of time—or even longer—than the single-threaded version!**

### Why There Is No Speedup:
* **No Parallelism:** Because only one thread can hold the GIL at a time, the physical math operations are still happening sequentially (one after another). You haven't distributed the load; you've just broken the load into tiny, fragmented shifts.
* **The Context Switch Tax:** Every time the threads switch places on the CPU, Python wastes micro-seconds saving their memory states, clearing caches, and trading the lock. This overhead actually adds extra time to your program.

---

## 📊 Summary Comparison: Your Two Code Experiments

| Experiment | Task Composition | How it behaves | Result |
| :--- | :--- | :--- | :--- |
| **Experiment 1 (Previous)** | 1 I/O-Bound (`input`) + 1 CPU-Bound | Truly concurrent. The input thread willingly gives up the GIL while waiting for you, letting the math thread run cleanly. | **🚀 Massive Time Savings (~4s vs ~7s)** |
| **Experiment 2 (Current)** | 2 CPU-Bound (`complex_calculation`) | Trapped. Both threads aggressively fight over the GIL, forcing the CPU to continuously switch back and forth. | **⏳ No Time Savings (Equal or Slower)** |

---

## 💡 How to make them run truly independently?
If you want these two `complex_calculation` tasks to run 100% independently on separate CPU cores at the exact same physical millisecond, you must swap out the `threading` module for the **`multiprocessing`** module we looked at earlier!

---
---